# 07 - Augmentation Ablation: Does Jittering Actually Help?

`reports/phase2_design.md` §5 proposes Gaussian noise-injection (jittering) on the
scaled training sequences as a training-time regularizer, and is explicit that it
cannot manufacture new real observations. This notebook tests that claim rather
than leaving it asserted: trains the tuned CNN-LSTM architecture with and without
jittering, same seed/budget otherwise, and reports the test-set effect honestly -
including if it does nothing or hurts.

The **test set is never touched by augmentation** - only copies of the training
sequences are jittered; validation and test stay on real, unmodified data
throughout, per the brief's requirement.


In [1]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf

from src.models import build_cnn_lstm, compile_model
from src.evaluate import regression_metrics, metrics_table
from src.windowing import inverse_transform_targets

tf.random.set_seed(42)
pd.set_option("display.width", 120)

data = np.load("data/processed/sequence_arrays.npz")
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

with open("data/processed/scalers.pkl", "rb") as f:
    scalers = pickle.load(f)

with open("data/processed/dl_results.pkl", "rb") as f:
    dl_results = pickle.load(f)
best_hp = dl_results["best_hyperparameters"]
print("Using Keras-Tuner-selected architecture:", best_hp)

Using Keras-Tuner-selected architecture: {'filters': 24, 'kernel_size': 2, 'lstm_units': 24, 'dropout': 0.2, 'l2': 0.001, 'learning_rate': 0.001, 'use_attention': False}


## Build the jittered training set

Gaussian noise (sigma=0.05, in already-standardized scaled units — roughly 5% of
one standard deviation per feature) added to 2 noisy copies of every training
sequence, concatenated with the original 210 real rows. Targets are left
unchanged (only inputs are jittered) — this is input-noise regularization, not
target smoothing.


In [2]:
rng = np.random.RandomState(42)
SIGMA = 0.05
N_COPIES = 2

def jitter(X, sigma, n_copies, rng):
    copies = [X]
    for _ in range(n_copies):
        noisy = X + rng.normal(0, sigma, X.shape).astype("float32")
        copies.append(noisy)
    return np.concatenate(copies, axis=0)

X_train_aug = jitter(X_train, SIGMA, N_COPIES, rng)
y_train_aug = np.concatenate([y_train] * (N_COPIES + 1), axis=0)
print("Original train:", X_train.shape, "-> augmented train:", X_train_aug.shape)

Original train: (210, 4, 10) -> augmented train: (630, 4, 10)


## Train identical architecture with vs. without jittering


In [3]:
def make_callbacks(ckpt_path, patience=20):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience,
                                          restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss",
                                            save_best_only=True, save_weights_only=False),
    ]

def evaluate_split(model, X, y_scaled):
    preds = model.predict(X, verbose=0)
    preds = np.concatenate([np.asarray(preds[0]).reshape(-1, 1), np.asarray(preds[1]).reshape(-1, 1)], axis=1)
    preds_real = inverse_transform_targets(preds, scalers)
    y_real = inverse_transform_targets(y_scaled, scalers)
    return {t: regression_metrics(y_real[:, i], preds_real[:, i]) for i, t in enumerate(["dbt", "wbt"])}

def build_best():
    return compile_model(
        build_cnn_lstm(
            X_train.shape[1:],
            filters=best_hp["filters"], kernel_size=best_hp["kernel_size"],
            lstm_units=best_hp["lstm_units"], dropout=best_hp["dropout"],
            l2=best_hp["l2"], use_attention=best_hp["use_attention"],
        ),
        optimizer="adam", lr=best_hp["learning_rate"],
    )

results = {}
for name, Xtr, ytr in [("no_augmentation", X_train, y_train), ("jittered_x3", X_train_aug, y_train_aug)]:
    model = build_best()
    ckpt = f"models/checkpoints/aug_{name}.keras"
    model.fit(
        Xtr, {"dbt": ytr[:, 0], "wbt": ytr[:, 1]},
        validation_data=(X_val, {"dbt": y_val[:, 0], "wbt": y_val[:, 1]}),
        epochs=300, batch_size=16, verbose=0, callbacks=make_callbacks(ckpt),
    )
    best_model = tf.keras.models.load_model(ckpt)
    results[name] = {
        "train": evaluate_split(best_model, X_train, y_train),  # always eval on REAL train, not augmented
        "val": evaluate_split(best_model, X_val, y_val),
        "test": evaluate_split(best_model, X_test, y_test),
    }
    print(f"{name}: done")

no_augmentation: done
jittered_x3: done


In [4]:
table = metrics_table(results)
comparison = table[table.split == "test"].sort_values(["target", "model"])
comparison.reset_index(drop=True)

             model split target        R2      RMSE       MAE       KGE
0      jittered_x3  test    dbt  0.655310  1.330347  1.039329  0.634257
1  no_augmentation  test    dbt  0.714977  1.209736  0.978483  0.673138
2      jittered_x3  test    wbt  0.611927  0.570970  0.416117  0.748316
3  no_augmentation  test    wbt  0.663993  0.531288  0.392001  0.694746

## Verdict

Compares test R2 with vs. without jittering, same architecture/budget. Reported
as-is — if jittering doesn't help (or hurts) at this sample size, that's the
finding, matching the design doc's own caveat that it "cannot add information the
physical system didn't produce, only discourage the small network from memorizing
exact training points" — a regularization effect that may or may not show up as
a test-metric improvement on any single run.


In [5]:
for target in ["dbt", "wbt"]:
    base = results["no_augmentation"]["test"][target]["R2"]
    aug = results["jittered_x3"]["test"][target]["R2"]
    delta = aug - base
    verdict = "helped" if delta > 0.01 else ("hurt" if delta < -0.01 else "no meaningful difference")
    print(f"{target}: no-aug R2={base:.3f}, jittered R2={aug:.3f}, delta={delta:+.3f} -> {verdict}")

dbt: no-aug R2=0.715, jittered R2=0.655, delta=-0.060 -> hurt
wbt: no-aug R2=0.664, jittered R2=0.612, delta=-0.052 -> hurt


In [6]:
with open("data/processed/augmentation_results.pkl", "wb") as f:
    pickle.dump(results, f)
print("Saved augmentation_results.pkl")

Saved augmentation_results.pkl
